In [ ]:
!pip -q install "transformers>=4.41.0" datasets accelerate peft bitsandbytes pillow wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.6 MB/s eta 0:00:00


In [ ]:
import os, wandb
wandb.login()

os.environ["WANDB_PROJECT"]  = "qwen-vl-arabic"
os.environ["WANDB_RUN_NAME"] = "qwenvl_lora_augmented"
os.environ["WANDB_LOG_MODEL"] = "false"

In [ ]:
import os, json
from datasets import Dataset

JSONL_PATH = "Arabic_LLava.jsonl"

IMAGE_DIRS = [
    "arabic_images",
    "CLEVR",
    "IconQA",
]

def resolve_image_path(p: str):
    if not p:
        return None

    if os.path.exists(p):
        return p

    for d in IMAGE_DIRS:
        cand = os.path.join(d, p)
        if os.path.exists(cand):
            return cand

    base = os.path.basename(p)
    for d in IMAGE_DIRS:
        cand = os.path.join(d, base)
        if os.path.exists(cand):
            return cand

    return None

rows = []
missing_images = 0

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)

        img_path = resolve_image_path(ex.get("image"))
        if img_path is None:
            missing_images += 1
            continue

        q = (ex.get("question") or "").strip()
        a = (ex.get("augmented_answer") or "").strip()

        if not q or not a:
            continue

        rows.append({
            "id": ex.get("id"),
            "image_path": img_path,
            "question": q,
            "augmented_answer": a
        })

print("Loaded:", len(rows))
print("Missing images:", missing_images)

dataset = Dataset.from_list(rows)
dataset

Loaded: 5018
Missing images: 0


Dataset({
    features: ['id', 'image_path', 'question', 'augmented_answer'],
    num_rows: 5018
})

In [ ]:
SEED = 42
train_ratio = 0.80
val_ratio   = 0.10
test_ratio  = 0.10

assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

# Shuffle then split
dataset = dataset.shuffle(seed=SEED)
n = len(dataset)
n_train = int(n * train_ratio)
n_val   = int(n * val_ratio)
n_test  = n - n_train - n_val

train_ds = dataset.select(range(0, n_train))
val_ds   = dataset.select(range(n_train, n_train + n_val))
test_ds  = dataset.select(range(n_train + n_val, n))

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

Train: 4014 Val: 501 Test: 503


In [ ]:
import os, json

OUT_DIR = "/content/splits"
os.makedirs(OUT_DIR, exist_ok=True)

def save_jsonl(ds, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in ds:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

save_jsonl(train_ds, os.path.join(OUT_DIR, "train.jsonl"))
save_jsonl(val_ds,   os.path.join(OUT_DIR, "val.jsonl"))
save_jsonl(test_ds,  os.path.join(OUT_DIR, "test.jsonl"))

print("Saved to:", OUT_DIR)

Saved to: /content/splits


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# ✅ load model with 4bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 8,297,212,928 || trainable%: 0.0608


In [ ]:
!pip -q install -U git+https://github.com/huggingface/transformers
!pip -q install -U qwen-vl-utils[decord]==0.0.8

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 89.2 MB/s eta 0:00:00


In [ ]:
from qwen_vl_utils import process_vision_info
from PIL import Image

MAX_LEN = 2048

def tokenize_example(example):
    try:
        image = Image.open(example["image_path"]).convert("RGB")
        question = example["question"]
        answer = example["augmented_answer"]

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }]

        image_inputs, video_inputs = process_vision_info(messages)
        prompt_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        full_text = prompt_text + answer

        full = processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )

        if "image_grid_thw" not in full:
            print(f"⚠️ Skipping example {example.get('id')} - missing image_grid_thw")
            return None

        prompt_enc = processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )

        input_ids = full["input_ids"][0]
        attn_mask = full["attention_mask"][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc["input_ids"].shape[1]
        labels[:min(prompt_len, labels.shape[0])] = -100

        # ✅ process pixel_values
        pixel_values = full["pixel_values"]
        if pixel_values.dim() >= 1 and pixel_values.shape[0] == 1:
            pixel_values = pixel_values.squeeze(0)

        # ✅ process grid_thw
        grid_thw = full["image_grid_thw"]
        if grid_thw.dim() == 2 and grid_thw.shape[0] == 1:
            grid_thw = grid_thw.squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "labels": labels,
            "pixel_values": pixel_values,
            "image_grid_thw": grid_thw.to(torch.long),
        }

    except Exception as e:
        print(f"❌ Error processing example {example.get('id')}: {e}")
        return None

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
import torch

@dataclass
class DataCollatorForQwen25VL:
    pad_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # ✅ remove null values
        features = [f for f in features if f is not None and "image_grid_thw" in f]

        if len(features) == 0:
            raise ValueError("Batch contains no valid examples!")

        #  transform to tensors
        for f in features:
            for k in ["input_ids", "attention_mask", "labels"]:
                if not isinstance(f[k], torch.Tensor):
                    f[k] = torch.tensor(f[k])
            for k in ["pixel_values", "image_grid_thw"]:
                if k in f and not isinstance(f[k], torch.Tensor):
                    f[k] = torch.tensor(f[k])

        max_len = max(f["input_ids"].shape[0] for f in features)

        def pad_1d(x, pad_value):
            pad_len = max_len - x.shape[0]
            if pad_len <= 0:
                return x
            return torch.cat([x, torch.full((pad_len,), pad_value, dtype=x.dtype)], dim=0)

        batch = {
            "input_ids": torch.stack([pad_1d(f["input_ids"], self.pad_token_id) for f in features]),
            "attention_mask": torch.stack([pad_1d(f["attention_mask"], 0) for f in features]),
            "labels": torch.stack([pad_1d(f["labels"], -100) for f in features]),
        }

        # ✅ Merge vision tensors correctly
        batch["pixel_values"] = torch.cat([f["pixel_values"] for f in features], dim=0)
        batch["image_grid_thw"] = torch.stack([f["image_grid_thw"] for f in features], dim=0)

        return batch

In [ ]:
# Re-process with filtering
train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names).filter(lambda x: x is not None)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names).filter(lambda x: x is not None)

print(f"✅ Train samples: {len(train_tok)}")
print(f"✅ Val samples: {len(val_tok)}")

# Create new collator
collator = DataCollatorForQwen25VL(pad_token_id=processor.tokenizer.pad_token_id)

# Test
try:
    sample_batch = collator([train_tok[0], train_tok[1]])
    print("✅ Collator test passed!")
    print(f"Batch keys: {sample_batch.keys()}")
except Exception as e:
    print(f"❌ Collator test failed: {e}")

Map:   0%|          | 0/4014 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4014 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

Filter:   0%|          | 0/501 [00:00<?, ? examples/s]

✅ Train samples: 4014
✅ Val samples: 501
✅ Collator test passed!
Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels', 'pixel_values', 'image_grid_thw'])


In [ ]:
tmp = tokenize_example(train_ds[0])
print(tmp.keys())
print("input_ids:", tmp["input_ids"].shape)
print("pixel_values:", tmp["pixel_values"].shape if "pixel_values" in tmp else None)
print("image_grid_thw:", tmp["image_grid_thw"] if "image_grid_thw" in tmp else None)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwenvl_lora_augmented",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    remove_unused_columns=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=1,
    report_to="wandb",
    run_name=os.environ["WANDB_RUN_NAME"],
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
400,0.604650,0.624493


TrainOutput(global_step=502, training_loss=0.671080669321387, metrics={'train_runtime': 15917.2513, 'train_samples_per_second': 0.504, 'train_steps_per_second': 0.032, 'total_flos': 2.151062134899548e+17, 'train_loss': 0.671080669321387, 'epoch': 2.0})

In [ ]:
trainer.save_model("./qwenvl_lora_augmented_adapter")
processor.save_pretrained("./qwenvl_lora_augmented_adapter")
print("Saved to ./qwenvl_lora_augmented_adapter")

Saved to ./qwenvl_lora_augmented_adapter


In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
                (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
                (up_proj): Linear4bit(

In [ ]:
import re
from tqdm import tqdm

# ✅ Dictionary for converting written Arabic numbers
ARABIC_NUMBERS = {
    'صفر': '0', 'واحد': '1', 'اثنان': '2', 'اثنين': '2', 'ثلاثة': '3', 'أربعة': '4',
    'خمسة': '5', 'ستة': '6', 'سبعة': '7', 'ثمانية': '8', 'تسعة': '9', 'عشرة': '10',
    'أحد عشر': '11', 'اثنا عشر': '12', 'ثلاثة عشر': '13', 'أربعة عشر': '14',
    'خمسة عشر': '15', 'ستة عشر': '16', 'سبعة عشر': '17', 'ثمانية عشر': '18',
    'تسعة عشر': '19', 'عشرون': '20', 'ثلاثون': '30', 'أربعون': '40',
    'خمسون': '50', 'ستون': '60', 'سبعون': '70', 'ثمانون': '80', 'تسعون': '90',
    'مئة': '100', 'مائة': '100', 'ألف': '1000',
}

def normalize_gt(text: str) -> str:
    """
    Enhanced normalization with support for written Arabic numbers
    """
    if not text:
        return ""

    text = str(text).strip()

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Convert written Arabic numbers to digits
    for word, num in ARABIC_NUMBERS.items():
        if word.lower() in text:
            text = num
            break

    # Remove punctuation marks
    text = re.sub(r'[.,،؛]', '', text)
    text = text.strip()

    return text

def extract_from_conclusion(text: str):
    """Extract text from inside <CONCLUSION> tags"""
    m = re.search(r"<CONCLUSION>(.*?)</CONCLUSION>", text, flags=re.S | re.I)
    if m:
        return m.group(1).strip()
    return None

def extract_pred(generated_text: str) -> str:
    """
    Extract prediction - supports numbers, text, and letters
    """
    if not generated_text:
        return None

    # 1. From CONCLUSION (highest priority)
    conc = extract_from_conclusion(generated_text)
    if conc:
        return conc

    # 2. Search for Arabic patterns
    patterns = [
        r'الإجابة(?:\s+(?:هي|النهائية|:))?\s*(.+?)(?:\.|$)',
        r'الجواب(?:\s+(?:هو|:))?\s*(.+?)(?:\.|$)',
        r'النتيجة(?:\s+(?:هي|:))?\s*(.+?)(?:\.|$)',
        r'الحل(?:\s+(?:هو|:))?\s*(.+?)(?:\.|$)',
    ]

    for pattern in patterns:
        match = re.search(pattern, generated_text, re.I)
        if match:
            result = match.group(1).strip()
            # Take only the first word/number
            result = result.split()[0] if result.split() else result
            return result

    # 3. Last number or word before end of text
    # Search for the last sentence
    sentences = generated_text.split('.')
    if sentences:
        last_sentence = sentences[-1].strip()
        words = last_sentence.split()
        if words:
            return words[-1]

    return None

def generate_one(model, processor, image_path, question, max_new_tokens=256):
    """
    Generate a single answer from the model
    """
    try:
        from PIL import Image
        from qwen_vl_utils import process_vision_info
        import torch

        image = Image.open(image_path).convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }]

        image_inputs, video_inputs = process_vision_info(messages)

        prompt_text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            padding=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        generated_text = processor.decode(generated_ids, skip_special_tokens=True)

        return generated_text.strip()

    except Exception as e:
        print(f"Error: {e}")
        return ""

def evaluate_final(model, processor, test_ds, max_samples=None):
    """
    Enhanced final evaluation
    """
    correct = 0
    total = 0
    examples_log = []

    samples = test_ds if max_samples is None else test_ds.select(range(min(max_samples, len(test_ds))))

    for ex in tqdm(samples, desc="Evaluating"):
        # ✅ Use augmented_answer
        gt_raw = ex.get("augmented_answer", "")
        if not gt_raw:
            continue

        # Extract from CONCLUSION in Ground Truth
        gt_from_conclusion = extract_from_conclusion(gt_raw)
        if not gt_from_conclusion:
            continue

        pred_text = generate_one(model, processor, ex["image_path"], ex["question"], max_new_tokens=256)
        pred_raw = extract_pred(pred_text)

        if not pred_raw:
            continue

        total += 1

        pred_norm = normalize_gt(pred_raw)
        gt_norm = normalize_gt(gt_from_conclusion)

        is_correct = pred_norm == gt_norm
        if is_correct:
            correct += 1

        if len(examples_log) < 20:
            examples_log.append({
                "id": ex.get("id", "")[:50],
                "question": ex["question"][:100],
                "gt_raw": gt_from_conclusion,
                "gt_norm": gt_norm,
                "pred_raw": pred_raw,
                "pred_norm": pred_norm,
                "full_output": pred_text[:200],
                "correct": is_correct
            })

    em = correct / total if total else 0.0

    return {
        "exact_match": em,
        "correct": correct,
        "total": total,
        "accuracy_percent": em * 100,
        "examples": examples_log
    }


print(f"\n{'='*60}")
print("🚀 Running FULL evaluation on all test data...")
print(f"{'='*60}")

metrics = evaluate_final(model, processor, test_ds, max_samples=None)

print(f"\n{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")
print(f"Exact Match: {metrics['exact_match']:.4f}")
print(f"Accuracy: {metrics['accuracy_percent']:.2f}%")
print(f"Correct: {metrics['correct']}/{metrics['total']}")

# Log to W&B
try:
    import wandb
    wandb.log({
        "test/exact_match_final": metrics["exact_match"],
        "test/accuracy_percent_final": metrics["accuracy_percent"],
        "test/correct_final": metrics["correct"],
        "test/total_final": metrics["total"],
    })

    # Log examples
    table = wandb.Table(
        columns=["ID", "Question", "GT", "Pred", "GT_Norm", "Pred_Norm", "Correct"],
        data=[
            [ex["id"], ex["question"][:80], ex["gt_raw"], ex["pred_raw"],
             ex["gt_norm"], ex["pred_norm"], "✅" if ex["correct"] else "❌"]
            for ex in metrics["examples"]
        ]
    )
    wandb.log({"test/predictions_final": table})

    print("\n✅ Logged to W&B!")
except Exception as e:
    print(f"\n⚠️ W&B logging failed: {e}")

print("\n" + "="*60)
print("Evaluation Complete!")
print("="*60)


🚀 Running FULL evaluation on all test data...


Evaluating: 100%|██████████| 503/503 [2:47:03<00:00, 19.93s/it]


FINAL RESULTS
Exact Match: 0.4334
Accuracy: 43.34%
Correct: 218/503

⚠️ W&B logging failed: You must call wandb.init() before wandb.log()

Evaluation Complete!


In [ ]:
from datasets import Dataset

test_ds = Dataset.from_list(load_jsonl("/content/splits/test.jsonl"))
print("="*60)
print("FINAL EVALUATION WITH CORRECT EXTRACTION")
print("="*60)

print("\n🔍 Quick test on 5 samples...")
metrics_quick = evaluate_final(base_model, processor, test_ds, max_samples=5)
print(f"\nQuick Results:")
print(f"  Accuracy: {metrics_quick['accuracy_percent']:.2f}%")
print(f"  Correct: {metrics_quick['correct']}/{metrics_quick['total']}")

print(f"\n{'='*60}")
print("SAMPLE PREDICTIONS (First 5):")
print(f"{'='*60}")
for i, ex in enumerate(metrics_quick['examples'][:5], 1):
    status = "✅" if ex['correct'] else "❌"
    print(f"\n{status} Example {i}:")
    print(f"  Q: {ex['question']}")
    print(f"  GT (raw): {ex['gt_raw']}")
    print(f"  GT (norm): {ex['gt_norm']}")
    print(f"  Pred (raw): {ex['pred_raw']}")
    print(f"  Pred (norm): {ex['pred_norm']}")
    if not ex['correct']:
        print(f"  Output: {ex['full_output']}")

FINAL EVALUATION WITH CORRECT EXTRACTION

🔍 Quick test on 20 samples...


Evaluating: 100%|██████████| 5/5 [01:36<00:00, 19.25s/it]


Quick Results:
  Accuracy: 40.00%
  Correct: 2/5

SAMPLE PREDICTIONS (First 5):

❌ Example 1:
  Q: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤال:


  GT (raw): سبعة عشر
  GT (norm): 7
  Pred (raw): الإجابة النهائية هي: خمسة عشر
  Pred (norm): 5
  Output: <SUMMARY>سأقوم بعدّ المستطيلات في الصورة وتحديد نوعها من خلال النظر إلى الترتيب واللون.</SUMMARY><CAPTION>تحتوي الصورة على شبكة من المستطيلات ملونة بألوان مختلفة: أخضر، أزرق، صفراء، بنفسجي، وبنفسجي فا

✅ Example 2:
  Q: يرجى الإجابة على السؤال أدناه، موضحًا استدلالك الخاص بك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤ
  GT (raw): 3
  GT (norm): 3
  Pred (raw): 3
  Pred (norm): 3

✅ Example 3:
  Q: يرجى الإجابة عن السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤال:

إ
  GT (raw): ب
  GT (norm): ب
  Pred (raw): ب
  Pred (norm): ب

❌ Example 4:
  Q: يرجى الإجابة على السؤال أدناه، مع شرح استنتاجك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤال:
ما هي
  GT (r

In [ ]:
import os
import gc
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

try:
    del model
    del base_model
except:
    pass
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
base_model.eval()

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("✅ Base model loaded!")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ Base model loaded!


In [ ]:

print(f"\n{'='*60}")
print("🚀 Running FULL evaluation on all test data...")
print(f"{'='*60}")

metrics = evaluate_final(base_model, processor, test_ds, max_samples=None)

print(f"\n{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")
print(f"Exact Match: {metrics['exact_match']:.4f}")
print(f"Accuracy: {metrics['accuracy_percent']:.2f}%")
print(f"Correct: {metrics['correct']}/{metrics['total']}")

# Log to W&B
try:
    import wandb
    wandb.log({
        "test/exact_match_final": metrics["exact_match"],
        "test/accuracy_percent_final": metrics["accuracy_percent"],
        "test/correct_final": metrics["correct"],
        "test/total_final": metrics["total"],
    })

    # Log examples
    table = wandb.Table(
        columns=["ID", "Question", "GT", "Pred", "GT_Norm", "Pred_Norm", "Correct"],
        data=[
            [ex["id"], ex["question"][:80], ex["gt_raw"], ex["pred_raw"],
             ex["gt_norm"], ex["pred_norm"], "✅" if ex["correct"] else "❌"]
            for ex in metrics["examples"]
        ]
    )
    wandb.log({"test/predictions_final": table})

    print("\n✅ Logged to W&B!")
except Exception as e:
    print(f"\n⚠️ W&B logging failed: {e}")

print("\n" + "="*60)
print("Evaluation Complete!")
print("="*60)


🚀 Running FULL evaluation on all test data...


Evaluating: 100%|██████████| 503/503 [2:01:02<00:00, 14.44s/it]


FINAL RESULTS
Exact Match: 0.0048
Accuracy: 0.48%
Correct: 2/413

⚠️ W&B logging failed: You must call wandb.init() before wandb.log()

Evaluation Complete!


In [ ]:
from datasets import Dataset

test_ds = Dataset.from_list(load_jsonl("/content/splits/test.jsonl"))

print("="*60)
print("FINAL EVALUATION WITH CORRECT EXTRACTION")
print("="*60)

print("\n🔍 Quick test on 5 samples...")
metrics_quick = evaluate_final(base_model, processor, test_ds, max_samples=5)
print(f"\nQuick Results:")
print(f"  Accuracy: {metrics_quick['accuracy_percent']:.2f}%")
print(f"  Correct: {metrics_quick['correct']}/{metrics_quick['total']}")

print(f"\n{'='*60}")
print("SAMPLE PREDICTIONS (First 5):")
print(f"{'='*60}")
for i, ex in enumerate(metrics_quick['examples'][:5], 1):
    status = "✅" if ex['correct'] else "❌"
    print(f"\n{status} Example {i}:")
    print(f"  Q: {ex['question']}")
    print(f"  GT (raw): {ex['gt_raw']}")
    print(f"  GT (norm): {ex['gt_norm']}")
    print(f"  Pred (raw): {ex['pred_raw']}")
    print(f"  Pred (norm): {ex['pred_norm']}")
    if not ex['correct']:
        print(f"  Output: {ex['full_output']}")

FINAL EVALUATION WITH CORRECT EXTRACTION

🔍 Quick test on 20 samples...


Evaluating: 100%|██████████| 5/5 [01:01<00:00, 12.30s/it]


Quick Results:
  Accuracy: 0.00%
  Correct: 0/3

SAMPLE PREDICTIONS (First 5):

❌ Example 1:
  Q: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤال:


  GT (raw): سبعة عشر
  GT (norm): 7
  Pred (raw): :
  Pred (norm): :
  Output: لحساب عدد المستطيلات في الصورة، سأقوم بالتدقيق في كل صف واحد بواحد:

1. الصف الأول: يحتوي على 5 مستطيلات.
2. الصف الثاني: يحتوي على 6 مستطيلات.
3. الصف الثالث: يحتوي على 3 مستطيلات.

إجمالي عدد المستط

❌ Example 2:
  Q: يرجى الإجابة على السؤال أدناه، مع شرح استنتاجك خطوة بخطوة قبل تقديم الإجابة النهائية.

السؤال:
ما هي
  GT (raw): الإجابة النهائية هي 2.
  GT (norm): الإجابة النهائية هي 2
  Pred (raw): :**
  Pred (norm): :**
  Output: لإجابة هذا السؤال، سنقوم بالتحليل الخطوة بخطوة:

1. **تحديد المجموعة:** نلاحظ أن السؤال يشير إلى مجموعة بيانات "الاسم". في الرسم البياني، هناك مربعين رئيسيين، أحدهما مخصص لمجموعة بيانات "الاسم" (مكتوب

❌ Example 3:
  Q: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة ق